In [39]:
from dotenv import load_dotenv
from openai import OpenAI
import os

load_dotenv()
openai_client = OpenAI(
    base_url="https://openrouter.ai/api/v1",
    # Automatically pulls the key from your .env file
    api_key=os.getenv("OPENROUTER_API_KEY"), 
)

In [40]:
from ingest import load_faq_data, build_index

documents = load_faq_data()
index = build_index(documents)

In [41]:
from rag_helper import RAGBase


instructions = """
You're a course teaching assistant.
Answer the QUESTION based on the CONTEXT from the FAQ database.
Use only the facts from the CONTEXT when answering the QUESTION.
""".strip()

assistant = RAGBase(
    index=index,
    llm_client=openai_client,
    instructions=instructions,
)

In [42]:
answer = assistant.rag('How do I run Olama locally?')
print(answer)

Yes—you can run the Olama (course) locally instead of using GitHub Codespaces. Codespaces is only the “one‑click” way to give everyone the same environment; it isn’t required.

To run it locally you’ll need to set up the development stack yourself, for example:

1. **Python** – install the desired version (the course expects a recent 3.x release).  
2. **uv** – the fast Python package manager used in the notebooks; install it (`curl -LsSf https://astral.sh/uv/install.sh | sh`).  
3. **Jupyter** – launch the notebooks that the course provides (`uv tool install jupyterlab` and then `jupyter lab`).  
4. **Docker** – some modules use containers (e.g., for vector stores or retrieval services), so have Docker Desktop or the Docker Engine running.  
5. **Any additional tools** mentioned in the module READMEs (e.g., `dirdotenv` for secret handling, `git`, etc.).

Once those pieces are installed:

```bash
# Clone the repo
git clone https://github.com/your‑org/olama-course.git
cd olama-course

#

In [43]:
answer = assistant.rag('How do I run Ollllama locally?')
print(answer)

I’m sorry, but the FAQ you provided doesn’t contain any information about running **Ollllama** locally. If you have a specific guide or documentation elsewhere, please share it, and I’ll be happy to help you interpret the steps!


In [44]:
messages = [
    {'role': 'user', 'content': 'I just discovered the course. Can I join it?'}
]

response = openai_client.responses.create(
    model='google/gemma-4-31b-it:free',
    input=messages,
)

response.output_text

'Since I am an AI, I don\'t know which specific course you are referring to! However, I can help you figure out how to join it.\n\n**To give you the right answer, could you tell me:**\n1. **What is the name of the course?**\n2. **Where did you find it?** (e.g., Coursera, Udemy, a specific university website, a YouTube description, or a social media ad).\n\n---\n\n### General Guide on how to join most online courses:\n\n**1. If it is an Open Enrollment course (Coursera, Udemy, edX):**\n*   Go to the course landing page.\n*   Click the **"Enroll," "Join for Free,"** or **"Buy Now"** button.\n*   Create an account and start learning immediately.\n\n**2. If it is a Cohort-Based course (Starts on a specific date):**\n*   Check the **"Enrollment Date."** If the window is open, you can sign up.\n*   If the course has already started, look for a **"Waitlist"** or a **"Join the next cohort"** button.\n\n**3. If it is a Private or University course:**\n*   Look for an **"Apply Now"** or **"Conta

In [45]:
def search(query):
    boost_dict = {'question': 3.0, 'section': 0.5}
    filter_dict = {'course': 'llm-zoomcamp'}

    return index.search(
        query,
        num_results=5,
        boost_dict=boost_dict,
        filter_dict=filter_dict
    )

In [46]:
search_tool = {
    "type": "function",
    'name': 'search',
    'description': 'Search the FAQ database for entries matching the given query.',
    'parameters': {
        "type": "object",
        "properties": {
            'query': {
                "type": "string",
                'description': 'Search query text to look up in the course FAQ.'
            }
        },
        "required": ["query"],
        'additionalProperties': False
    }
}

In [47]:
response = openai_client.responses.create(
    model='google/gemma-4-31b-it:free',
    input=messages,
    tools=[search_tool]
)

In [48]:
call = response.output[0]

In [49]:
call

ResponseFunctionToolCall(arguments='{"query": "joining the course"}', call_id='chatcmpl-tool-a9e85909987fba83', name='search', type='function_call', id='fc_tmp_sgj8kzyeb6', namespace=None, status='completed')

In [50]:
import json

args = json.loads(call.arguments)
args

{'query': 'joining the course'}

In [51]:
call.name

'search'

In [52]:
results = search(**args)

In [53]:
result_json = json.dumps(results, indent=2)

In [54]:
function_call_output = {
    "type": "function_call_output",
    'call_id': call.call_id,
    'output': result_json,
}

In [55]:
messages.append(call)

In [56]:
messages.append(function_call_output)

In [57]:
messages

[{'role': 'user', 'content': 'I just discovered the course. Can I join it?'},
 ResponseFunctionToolCall(arguments='{"query": "joining the course"}', call_id='chatcmpl-tool-a9e85909987fba83', name='search', type='function_call', id='fc_tmp_sgj8kzyeb6', namespace=None, status='completed'),
 {'type': 'function_call_output',
  'call_id': 'chatcmpl-tool-a9e85909987fba83',
  'output': '[\n  {\n    "id": "04919992b3",\n    "course": "llm-zoomcamp",\n    "section": "General Course-Related Questions",\n    "question": "How should I start the course and follow the weekly workflow?",\n    "answer": "Start with the [LLM Zoomcamp docs](https://datatalks.club/docs/courses/llm-zoomcamp/), the [general Zoomcamp logistics docs](https://datatalks.club/docs/courses/zoomcamp-logistics/), and the [LLM Zoomcamp GitHub repository](https://github.com/DataTalksClub/llm-zoomcamp).\\n\\nYou can start whenever you want. The videos and GitHub materials are available, and the deadlines are listed in the [course man

In [58]:
response = openai_client.responses.create(
    model='google/gemma-4-31b-it:free',
    input=messages,
    tools=[search_tool]
)

In [59]:
print(response.output_text)

Yes, you can definitely join! You can start whenever you want.

If you are interested in earning a certificate, please keep in mind that you will need to submit your project while the course is still accepting submissions.

To get started, you can check out these resources:
*   **Course Documentation:** [LLM Zoomcamp docs](https://datatalks.club/docs/courses/llm-zoomcamp/)
*   **General Logistics:** [Zoomcamp logistics docs](https://datatalks.club/docs/courses/zoomcamp-logistics/)
*   **GitHub Repository:** [LLM Zoomcamp GitHub](https://github.com/DataTalksClub/llm-zoomcamp)
*   **Course Platform:** [Course management platform](https://courses.datatalks.club/llm-zoomcamp-2026/) (where you can find deadlines)


In [60]:
usage = response.usage
usage.input_tokens, usage.output_tokens

(831, 195)

In [61]:
def make_call(call):
    args = json.loads(call.arguments)

    if call.name == 'search':
        result = search(**args)

    result_json = json.dumps(result, indent=2)

    return {
        "type": "function_call_output",
        'call_id': call.call_id,
        'output': result_json,
    }

In [62]:
instructions = """
You're a course teaching assistant.
You're given a question from a course student and your task is to answer it.

If you want to look up information, use the search function. 
Use as many keywords from the user question as possible when making first requests.

Make multiple searches.

Try to expand your search by using new keywords
based on the results you get from the search.

At the end, ask if there are other areas that the user wants to explore.
"""

question = 'I just discovered the course. Can I join it?'


messages = [
    {'role': 'developer', 'content': instructions},
    {'role': 'user', 'content': question}
]

In [63]:
response = openai_client.responses.create(
    model='google/gemma-4-31b-it:free',
    input=messages,
    tools=[search_tool]
)

In [64]:
messages.extend(response.output)

for item in response.output:
    if item.type == 'function_call':
        print('function_call:', item.name, item.arguments)
        call_output = make_call(item)
        messages.append(call_output)

    elif item.type == 'message':
        print('ASSISTANT:')
        print(item.content[0].text)

function_call: search {"query": "can I join the course late? registration enrollment deadlines"}


In [65]:
messages

[{'role': 'developer',
  'content': "\nYou're a course teaching assistant.\nYou're given a question from a course student and your task is to answer it.\n\nIf you want to look up information, use the search function. \nUse as many keywords from the user question as possible when making first requests.\n\nMake multiple searches.\n\nTry to expand your search by using new keywords\nbased on the results you get from the search.\n\nAt the end, ask if there are other areas that the user wants to explore.\n"},
 {'role': 'user', 'content': 'I just discovered the course. Can I join it?'},
 ResponseFunctionToolCall(arguments='{"query": "can I join the course late? registration enrollment deadlines"}', call_id='chatcmpl-tool-91fa1e23f12ca8f3', name='search', type='function_call', id='fc_tmp_8qzj2f4pqlr', namespace=None, status='completed'),
 {'type': 'function_call_output',
  'call_id': 'chatcmpl-tool-91fa1e23f12ca8f3',
  'output': '[\n  {\n    "id": "74eb249bbf",\n    "course": "llm-zoomcamp",\n

In [67]:
messages = [
    {'role': 'developer', 'content': instructions},
    {'role': 'user', 'content': question}
]

it = 1

while True:
    print(f'iteration #{it}...')
    has_function_calls = False

    response = openai_client.responses.create(
        model='google/gemma-4-31b-it:free',
        input=messages,
        tools=[search_tool]
    )

    messages.extend(response.output)

    for item in response.output:
        if item.type == 'function_call':
            print('function_call:', item.name, item.arguments)
            call_output = make_call(item)
            messages.append(call_output)
            has_function_calls = True

        elif item.type == 'message':
            print('ASSISTANT:')
            print(item.content[0].text)
    
    it = it + 1
    if has_function_calls == False:
        break

iteration #1...


function_call: search {"query": "how to join the course enrollment registration"}
iteration #2...
ASSISTANT:
Yes, you can! You are welcome to join the course at any time.

Here are a few key things to know to get started:

*   **Getting Started:** You can begin learning and working through the materials immediately. You don't even need a confirmation email to start. I recommend checking out the following resources:
    *   [LLM Zoomcamp Docs](https://datatalks.club/docs/courses/llm-zoomcamp/)
    *   [General Zoomcamp Logistics Docs](https://datatalks.club/docs/courses/zoomcamp-logistics/)
    *   [LLM Zoomcamp GitHub Repository](https://github.com/DataTalksClub/llm-zoomcamp)
*   **Certificates:** If you are aiming to receive a certificate, please keep in mind that you must submit your project while submissions are still being accepted.
*   **Workflow:** The typical way to progress through the course is:
    1. Watch the lesson videos.
    2. Work through the lesson notebooks and code.

In [68]:
def agent_loop(instructions, question, model="google/gemma-4-31b-it:free") -> str:
    messages = [
        {"role": "developer", "content": instructions},
        {"role": "user", "content": question}
    ]

    it = 1

    while True:
        print(f"iteration #{it}...")
        has_function_calls = False

        response = openai_client.responses.create(
            model=model,
            input=messages,
            tools=[search_tool]
        )

        messages.extend(response.output)

        for item in response.output:
            if item.type == "function_call":
                print("function_call:", item.name, item.arguments)
                call_output = make_call(item)
                messages.append(call_output)
                has_function_calls = True

            elif item.type == "message":
                print("ASSISTANT:")
                last_answer = item.content[0].text
                print(item.content[0].text)

        it = it + 1
        if has_function_calls == False:
            break

    return last_answer

In [69]:
agent_loop(instructions, "How do I run Olama locally?")

iteration #1...
function_call: search {"query": "how to run Ollama locally"}
iteration #2...
ASSISTANT:
To run Ollama locally, follow these steps based on your operating system:

### 1. Installation
Visit [ollama.com/download](https://ollama.com/download) and select your OS:
*   **macOS**: Download and install the `.pkg` file.
*   **Windows**: Download and install the `.msi` file.
*   **Linux**: Run the following command in your terminal:
    ```bash
    curl -fsSL https://ollama.com/install.sh | sh
    ```

### 2. Running a Model
Once installed, open your terminal and run a model (for example, LLaMA 3):
```bash
ollama run llama3
```
This command will automatically download the model (approx. 4GB) and open an interactive chat interface.

### 3. Verifying the Server
You can verify that the Ollama server is running by sending a request to its local endpoint:
```bash
curl http://localhost:11434
```
A successful connection will return a response (e.g., `{"models": [...]}`).

### 4. Using w

'To run Ollama locally, follow these steps based on your operating system:\n\n### 1. Installation\nVisit [ollama.com/download](https://ollama.com/download) and select your OS:\n*   **macOS**: Download and install the `.pkg` file.\n*   **Windows**: Download and install the `.msi` file.\n*   **Linux**: Run the following command in your terminal:\n    ```bash\n    curl -fsSL https://ollama.com/install.sh | sh\n    ```\n\n### 2. Running a Model\nOnce installed, open your terminal and run a model (for example, LLaMA 3):\n```bash\nollama run llama3\n```\nThis command will automatically download the model (approx. 4GB) and open an interactive chat interface.\n\n### 3. Verifying the Server\nYou can verify that the Ollama server is running by sending a request to its local endpoint:\n```bash\ncurl http://localhost:11434\n```\nA successful connection will return a response (e.g., `{"models": [...]}`).\n\n### 4. Using with Python\nIf you want to integrate Ollama into your code, install the Python

In [71]:
agent_loop(instructions, "I just discovered the course. Can I still join it?")

iteration #1...


function_call: search {"query": "Can I still join the course? late enrollment registration deadline"}
iteration #2...
ASSISTANT:
Yes, you can still join the course! You can start learning and working through the materials whenever you like.

Here are a few important details regarding your participation:

*   **Joining & Registration:** You can start learning and submitting homework immediately (as long as the submission forms are open) without needing a formal registration confirmation.
*   **Certification:** If you are aiming to receive a certificate, you must submit your Capstone project while the course is still accepting submissions. Please note that certificates are only awarded to those in the "live" cohort, as the process requires peer-reviewing other students' projects.
*   **Homework:** While homework is not mandatory for the certificate, it is highly recommended to reinforce the concepts and helps you rank on the leaderboard.

**To get started, I recommend checking out these 

'Yes, you can still join the course! You can start learning and working through the materials whenever you like.\n\nHere are a few important details regarding your participation:\n\n*   **Joining & Registration:** You can start learning and submitting homework immediately (as long as the submission forms are open) without needing a formal registration confirmation.\n*   **Certification:** If you are aiming to receive a certificate, you must submit your Capstone project while the course is still accepting submissions. Please note that certificates are only awarded to those in the "live" cohort, as the process requires peer-reviewing other students\' projects.\n*   **Homework:** While homework is not mandatory for the certificate, it is highly recommended to reinforce the concepts and helps you rank on the leaderboard.\n\n**To get started, I recommend checking out these resources:**\n*   The [LLM Zoomcamp docs](https://datatalks.club/docs/courses/llm-zoomcamp/)\n*   The [general Zoomcamp

In [72]:
instructions = """
You're a course teaching assistant.
You're given a question from a course student and your task is to answer it.

If you want to look up information, use the search function. 
Use as many keywords from the user question as possible when making first requests.

Make multiple searches. First perform search, analyze the results 
and then perform more searches. 

At the end, ask if there are other areas that the user wants to explore.
""".strip()

agent_loop(instructions, "I just discovered the course. Can I join it?")

iteration #1...
function_call: search {"query": "joining the course late enrollment registration"}
iteration #2...
ASSISTANT:
Yes, you can absolutely join the course!

You can start whenever you want since the lesson videos and GitHub materials are always available. You don't even need a confirmation email to begin learning and submitting homework.

**A few important things to keep in mind:**

*   **Certificates:** If you are interested in receiving a certificate, you must submit your project while the course is still actively accepting submissions. Certificates are not awarded for self-paced learning because the process requires you to peer-review three capstone projects, which can only be done while the "live" cohort is running.
*   **Getting Started:** To begin, I recommend checking out the [LLM Zoomcamp docs](https://datatalks.club/docs/courses/llm-zoomcamp/), the [general Zoomcamp logistics docs](https://datatalks.club/docs/courses/zoomcamp-logistics/), and the [GitHub repository]

'Yes, you can absolutely join the course!\n\nYou can start whenever you want since the lesson videos and GitHub materials are always available. You don\'t even need a confirmation email to begin learning and submitting homework.\n\n**A few important things to keep in mind:**\n\n*   **Certificates:** If you are interested in receiving a certificate, you must submit your project while the course is still actively accepting submissions. Certificates are not awarded for self-paced learning because the process requires you to peer-review three capstone projects, which can only be done while the "live" cohort is running.\n*   **Getting Started:** To begin, I recommend checking out the [LLM Zoomcamp docs](https://datatalks.club/docs/courses/llm-zoomcamp/), the [general Zoomcamp logistics docs](https://datatalks.club/docs/courses/zoomcamp-logistics/), and the [GitHub repository](https://github.com/DataTalksClub/llm-zoomcamp).\n*   **Workflow:** A typical way to progress is:\n    1. Watch the l

In [73]:
agent_loop(instructions, "what is fifa world cup")

iteration #1...
function_call: search {"query": "what is fifa world cup"}
iteration #2...
ASSISTANT:
The FIFA World Cup is an international association football competition contested by the national teams of the members of FIFA, the governing body of association football. It is the most prestigious football tournament in the world and is held every four years.

*(Note: As a course TA for the LLM Zoomcamp, I should mention that this information is general knowledge and is not part of the course curriculum, which focuses on Large Language Models, RAG, and Vector Search.)*

Are there any other areas—perhaps related to the course materials—that you would like to explore?


'The FIFA World Cup is an international association football competition contested by the national teams of the members of FIFA, the governing body of association football. It is the most prestigious football tournament in the world and is held every four years.\n\n*(Note: As a course TA for the LLM Zoomcamp, I should mention that this information is general knowledge and is not part of the course curriculum, which focuses on Large Language Models, RAG, and Vector Search.)*\n\nAre there any other areas—perhaps related to the course materials—that you would like to explore?'

In [74]:
instructions = """
You're a course teaching assistant.
You're given a question from a course student and your task is to answer it.

If you want to look up information, use the search function. 
Use as many keywords from the user question as possible when making first requests.

Make multiple searches. First perform search, analyze the results 
and then perform more searches. 

The question has to be about the course or its logistics, offtopic questions 
shouldn't be answered. If the search returns nothing, it's likely an off-topic question.
If you can't answer the question using FAQ, don't do it yourself. Only use the 
facts from the FAQ database.

At the end, ask if there are other areas that the user wants to explore.
""".strip()

agent_loop(instructions, "what is fifa world cup")

iteration #1...
function_call: search {"query": "what is fifa world cup"}
iteration #2...
ASSISTANT:
I'm sorry, but I can only answer questions related to the course or its logistics. Since I couldn't find any information about the FIFA World Cup in the course materials, I cannot answer this question.

Are there any other course-related areas you would like to explore?


"I'm sorry, but I can only answer questions related to the course or its logistics. Since I couldn't find any information about the FIFA World Cup in the course materials, I cannot answer this question.\n\nAre there any other course-related areas you would like to explore?"